<a href="https://colab.research.google.com/github/Rogerio-mack/work/blob/main/Rule_Based_Programming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Expert Systems, Rule‑Based Programming**

Dmitry Soshnikov, Microsoft. Knowledge Representation and Expert Systems.
https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/2-Symbolic/README.md

# Sistemas Especialistas

Um dos primeiros sucessos da IA ​​simbólica foram os chamados **sistemas especialistas** - sistemas de computador que foram projetados para atuar como um especialista em algum domínio de problema limitado. Eles são baseados em uma base de conhecimento extraída de um ou mais especialistas e contêm um mecanismo de inferência que realiza algum tipo raciocínio sobre essa base.

Existem várias formas de representar o conhecimento. Representações de rede, por exemplo, são baseadas no fato de que temos uma rede de conceitos inter-relacionados e podemos tentar reproduzir isso com um grafo em um computador - isto é uma rede *semântica*.

<br>

<p>
  <img src="https://github.com/microsoft/AI-For-Beginners/raw/main/lessons/2-Symbolic/images/AND-OR-Tree.png" width="80%" align="left"/>
</p>

<br>



Este diagrama é chamado de árvore AND-OR , e é uma representação gráfica de um conjunto de **regras de produção**. Desenhar uma árvore é útil no início da extração de conhecimento do especialista. Para representar o conhecimento dentro do computador, entretanto, pode ser mais conveniente usar regras:

```
IF the animal eats meat
OR (animal has sharp teeth
    AND animal has claws
    AND animal has forward-looking eyes
)
THEN the animal is a carnivore
```

Você pode notar que cada condição no lado esquerdo da regra e a ação são essencialmente triplas objeto-atributo-valor (OAV).

# Inferência para frente vs. para trás

O processo descrito acima é chamado de **forward inference**, ou **inferência direta**. Ele começa com alguns dados iniciais sobre o problema disponíveis na memória de trabalho e, em seguida, executa o seguinte loop de raciocínio:

0. Inicie a memória de trabalho as regras desejadas.
1. Se o atributo alvo estiver presente memória de trabalho - pare e forneça o resultado
2. Procure todas as regras cuja condição seja atualmente satisfeita - obtenha um conjunto de regras de conflito .
3. Execute a resolução de conflitos - selecione uma regra que será executada nesta etapa. Pode haver diferentes estratégias de resolução de conflitos:
  - Selecione a primeira regra aplicável na base de conhecimento
  - Selecione uma regra aleatória
  - Selecione uma regra mais específica , ou seja, aquela que atende a mais condições no "lado esquerdo" (LHS)
4. Aplicar a regra selecionada e inserir um novo conhecimento no estado do problema
5. Repita a partir do passo 1.

Em alguns casos, entretanto, podemos querer começar com um conhecimento vazio sobre o problema e fazer perguntas que nos ajudarão a chegar à conclusão. Por exemplo, ao fazer um diagnóstico médico, geralmente não realizamos todas as análises médicas com antecedência antes de começar a diagnosticar o paciente. Em vez disso, queremos realizar análises quando uma decisão precisa ser tomada.Neste caso, o processo pode ser modelado usando **inferência reversa**. Ele é conduzido pelo objetivo - o valor do atributo que estamos procurando encontrar.

Aqui, trataremos unicamente da **inferência direta**, mas você pode consultar as referências e ver a solução de um problema de inferência reversa.

# **CLIPS, PyKnow, Rule‑Based Programming**

[**PyKnow**](https://github.com/buguroo/pyknow/) é uma biblioteca para criar sistemas de inferência direta em Python projetada para ser similar ao antigo sistema clássico [CLIPS](http://www.clipsrules.net/index.html).

Você poderia fazer uma implementação direta sem muitos problemas, mas implementações ingênuas geralmente não são eficientes e o PyKnow (e o CLIPS) implementa um algoritmo especial de inferência, o [Rete](https://en.wikipedia.org/wiki/Rete_algorithm), bastante eficiente.

In [ ]:
!pip uninstall yfinance # incompatível com o pyknow

Found existing installation: yfinance 0.2.54
Uninstalling yfinance-0.2.54:
  Would remove:
    /usr/local/bin/sample
    /usr/local/lib/python3.11/dist-packages/yfinance-0.2.54.dist-info/*
    /usr/local/lib/python3.11/dist-packages/yfinance/*
Proceed (Y/n)? Y
  Successfully uninstalled yfinance-0.2.54


In [ ]:
import sys
!{sys.executable} -m pip install git+https://github.com/buguroo/pyknow/

  Cloning https://github.com/buguroo/pyknow/ to /tmp/pip-req-build-wa6k32gd
  Running command git clone --filter=blob:none --quiet https://github.com/buguroo/pyknow/ /tmp/pip-req-build-wa6k32gd
  Resolved https://github.com/buguroo/pyknow/ to commit 48818336f2e9a126f1964f2d8dc22d37ff800fe8
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for pyknow: filename=pyknow-1.7.0-py3-none-any.whl size=34226 sha256=9ff32cb4c323f3bc48fe549127f5dba8de5e3efa2dc69ed066f4b5fe33604675
  Stored in directory: /tmp/pip-ephem-wheel-cache-__4vvkmz/wheels/81/1a/d3/f6c15dbe1955598a37755215f2a10449e7418500d7bd4b9508
  Created wheel for frozendict: filename=frozendict-1.2-py3-none-any.whl size=3149 sha256=042a70a3163d53086acb0e1b0c66e661c5377a31441f9c127f507c1011d79b75
  Stored in directory: /root/.cache/pip/wheels/49/ac/f8/cb8120244e710bdb479c86198b03c7b08c3c2d3d2bf448fd6e
Successfully built pyknow frozendict
  Attempting uninstall: frozendict
    Found existin

In [ ]:
import collections.abc
collections.Mapping = collections.abc.Mapping

from pyknow import *

Definimos nosso sistema como uma classe que subclassifica `KnowledgeEngine`. Cada regra é definida por uma função separada com a anotação `@Rule`, que especifica quando a regra deve disparar. Dentro da regra, podemos adicionar novos fatos usando a função `declare`. Adicionar esses fatos resultará em mais regras sendo chamadas pelo mecanismo de inferência direta.



## Exemplo: Sócrates é Mortal!

In [ ]:
class Humans(KnowledgeEngine):
    @Rule(Fact('human'))
    def mortal(self):
        self.declare(Fact('mortal'))

    @Rule(Fact('socrates'))
    def human(self):
        self.declare(Fact('human'))

    def factz(self,l):
      for x in l:
          self.declare(x)

Uma vez que definimos uma base de conhecimento, preenchemos nossa memória de trabalho com alguns fatos iniciais e, então, chamamos o método `run()` para executar a inferência. Você pode ver como resultado que novos fatos inferidos são adicionados à memória de trabalho, incluindo o fato final sobre o animal (se configurarmos todos os fatos iniciais corretamente).

In [ ]:
ex = Humans()
ex.reset()
ex.factz([Fact('socrates')])
ex.run()
ex.facts

FactList([(0, InitialFact()),
          (1, Fact('socrates')),
          (2, Fact('human')),
          (3, Fact('mortal'))])

# Um exemplo mais completo: Taxonomia Animal

In [ ]:
class Animals(KnowledgeEngine):
    @Rule(OR(
           AND(Fact('sharp teeth'),Fact('claws'),Fact('forward looking eyes')),
           Fact('eats meat')))
    def cornivor(self):
        self.declare(Fact('carnivor'))

    @Rule(OR(Fact('hair'),Fact('gives milk')))
    def mammal(self):
        self.declare(Fact('mammal'))

    @Rule(Fact('mammal'),
          OR(Fact('has hooves'),Fact('chews cud')))
    def hooves(self):
        self.declare('ungulate')

    @Rule(OR(Fact('feathers'),AND(Fact('flies'),Fact('lays eggs'))))
    def bird(self):
        self.declare('bird')

    @Rule(Fact('mammal'),Fact('carnivor'),
          Fact(color='red-brown'),
          Fact(pattern='dark spots'))
    def monkey(self):
        self.declare(Fact(animal='monkey'))

    @Rule(Fact('mammal'),Fact('carnivor'),
          Fact(color='red-brown'),
          Fact(pattern='dark stripes'))
    def tiger(self):
        self.declare(Fact(animal='tiger'))

    @Rule(Fact('ungulate'),
          Fact('long neck'),
          Fact('long legs'),
          Fact(pattern='dark spots'))
    def giraffe(self):
        self.declare(Fact(animal='giraffe'))

    @Rule(Fact('ungulate'),
          Fact(pattern='dark stripes'))
    def zebra(self):
        self.declare(Fact(animal='zebra'))

    @Rule(Fact('bird'),
          Fact('long neck'),
          Fact('cannot fly'),
          Fact(color='black and white'))
    def straus(self):
        self.declare(Fact(animal='ostrich'))

    @Rule(Fact('bird'),
          Fact('swims'),
          Fact('cannot fly'),
          Fact(color='black and white'))
    def pinguin(self):
        self.declare(Fact(animal='pinguin'))

    @Rule(Fact('bird'),
          Fact('flies well'))
    def albatros(self):
        self.declare(Fact(animal='albatross'))

    @Rule(Fact(animal=MATCH.a))
    def print_result(self,a):
          print('Animal is {}'.format(a))

    def factz(self,l):
        for x in l:
            self.declare(x)

In [ ]:
ex1 = Animals()
ex1.reset()
ex1.factz([
    Fact(color='red-brown'),
    Fact(pattern='dark stripes'),
    Fact('sharp teeth'),
    Fact('claws'),
    Fact('forward looking eyes'),
    Fact('gives milk')])
ex1.run()
ex1.facts

Animal is tiger


FactList([(0, InitialFact()),
          (1, Fact(color='red-brown')),
          (2, Fact(pattern='dark stripes')),
          (3, Fact('sharp teeth')),
          (4, Fact('claws')),
          (5, Fact('forward looking eyes')),
          (6, Fact('gives milk')),
          (7, Fact('mammal')),
          (8, Fact('carnivor')),
          (9, Fact(animal='tiger'))])